# Original vs reconstruction: M20, Gini, C, A
The goal of this notebook is to reproduce **Figure 4** of
[Euclid Collaboration: Csizi et al. (2025), *Euclid preparation. LXVII. Deep learning true galaxy
morphologies for weak lensing shear bias calibration*, A&A 695, A283](https://doi.org/10.1051/0004-6361/202452129)
([arXiv:2409.07528](https://arxiv.org/abs/2409.07528)): a galaxy-by-galaxy comparison of morphological
proxies measured on the original images (x-axis) and on their reconstructions by the generative
model (y-axis).

It reads the FITS catalogues written by `run_morphometrics.py`
(`catalog_real.fits` and `catalog_reconstruction.fits`) and draws panels (a)–(d): $M_{20}$, Gini $G$,
concentration $C$ and asymmetry $A$. Smoothness $S$ (panel e) is not computed by this pipeline, and the
example galaxies of panel (f) are left out.

Unlike the paper's log-scaled density, the colour shows the **fraction of objects** per hexagon on a
linear scale, and black contours enclose **68 % and 95 %** of the objects. Both are independent of the
sample size, so the scatter is not visually inflated when the catalogue is large.

In [ ]:
# !pip install -q astropy   # usually already installed on Colab

import os

import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table

In [ ]:
# --- Load the FITS catalogues ------------------------------------------------
# Option A (Colab): upload both files by hand       -> USE_UPLOAD = True
# Option B (Colab): read them from Google Drive     -> USE_DRIVE = True and set DATA_DIR below
# Option C (local): set DATA_DIR to the folder holding the catalogues (Colab cannot see local paths)
USE_UPLOAD = True
USE_DRIVE = False
DATA_DIR = "."

REAL_FILE = "catalog_real.fits"
RECO_FILE = "catalog_reconstruction.fits"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and USE_UPLOAD:
    from google.colab import files
    files.upload()  # select catalog_real.fits and catalog_reconstruction.fits
    DATA_DIR = "."
elif IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/results_droppedDB_newflow"

real = Table.read(os.path.join(DATA_DIR, REAL_FILE))
reco = Table.read(os.path.join(DATA_DIR, RECO_FILE))
print(len(real), "original objects,", len(reco), "reconstructions")

In [ ]:
# --- Matching and selection --------------------------------------------------
# Both catalogues are built from the same stamps: objects are matched by IDENT.
if not np.array_equal(np.asarray(real["IDENT"]), np.asarray(reco["IDENT"])):
    order = np.argsort(np.asarray(reco["IDENT"]))
    idx = order[np.searchsorted(np.asarray(reco["IDENT"])[order], np.asarray(real["IDENT"]))]
    reco = reco[idx]
    assert np.array_equal(np.asarray(real["IDENT"]), np.asarray(reco["IDENT"]))

# An object is kept if the (R) morphology measurement succeeded on BOTH the original and the reconstruction.
# -9 is the sentinel value written by the pipeline when a measurement fails.
base_mask = np.asarray(real["flag_morph"]) & np.asarray(reco["flag_morph"])


def valid(col):
    x = np.asarray(real[col], dtype=float)
    y = np.asarray(reco[col], dtype=float)
    m = base_mask & np.isfinite(x) & np.isfinite(y) & (x != -9) & (y != -9)
    return x[m], y[m]

In [ ]:
# --- Panel settings ----------------------------------------------------------
# limits=None -> automatic limits (0.5-99.5 percentiles of both samples)
# Limits used in the paper: M20 (-2.5, -0.5), Gini (0, 0.65), C (1, 4), A (-0.05, 0.4)
PANELS = [
    dict(col="M20",  label=r"(a) $M_{20}$ coefficient",  limits=None),
    dict(col="Gini", label=r"(b) Gini coefficient $G$",  limits=None),
    dict(col="C",    label=r"(c) Concentration $C$",     limits=None),
    dict(col="A",    label=r"(d) Asymmetry $A$",         limits=None),
]
XLABEL = "original"
YLABEL = "model reconstruction"
GRIDSIZE = 60
HIST_BINS = 30
CONTOUR_FRACTIONS = (0.95, 0.68)  # fraction of objects enclosed by each contour
CONTOUR_BINS = 40


def auto_limits(x, y, lo=0.5, hi=99.5, pad=0.05):
    v = np.concatenate([x, y])
    a, b = np.percentile(v, [lo, hi])
    d = (b - a) * pad
    return a - d, b + d


def enclosed_levels(H, fractions):
    """Histogram levels whose contours enclose the given fractions of objects."""
    h = np.sort(H.ravel())[::-1]
    cum = np.cumsum(h) / h.sum()
    return sorted(h[np.searchsorted(cum, f)] for f in fractions)


def scatter_panel(ax, x, y, label, limits):
    lim = limits if limits is not None else auto_limits(x, y)
    in_box = (x >= lim[0]) & (x <= lim[1]) & (y >= lim[0]) & (y <= lim[1])
    xs, ys = x[in_box], y[in_box]

    # Colour = fraction of objects per hexagon (linear, independent of sample size)
    w = np.full(len(xs), 1.0 / len(xs))
    hb = ax.hexbin(xs, ys, C=w, reduce_C_function=np.sum, gridsize=GRIDSIZE,
                   extent=(*lim, *lim), cmap="Blues", mincnt=1, linewidths=0)
    hb.set_clim(0, np.percentile(np.ma.compressed(hb.get_array()), 99.5))

    # Contours enclosing 68 % and 95 % of the objects
    H, xe, ye = np.histogram2d(xs, ys, bins=CONTOUR_BINS, range=[lim, lim])
    xc, yc = 0.5 * (xe[1:] + xe[:-1]), 0.5 * (ye[1:] + ye[:-1])
    ax.contour(xc, yc, H.T, levels=enclosed_levels(H, CONTOUR_FRACTIONS),
               colors="k", linewidths=[0.8, 1.3])

    ax.plot(lim, lim, ls=":", color="k", lw=1.2)
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.set_aspect("equal")
    ax.text(0.04, 0.95, label, transform=ax.transAxes, va="top", ha="left", fontsize=12)
    ax.set_xlabel(XLABEL, fontsize=12)
    ax.set_ylabel(YLABEL, fontsize=12)
    ax.tick_params(direction="in", top=True, right=True)

    # Marginal histograms: original on top, reconstruction on the right
    bins = np.linspace(*lim, HIST_BINS + 1)
    ax_top = ax.inset_axes([0, 1.0, 1, 0.15], sharex=ax)
    ax_top.hist(x[(x >= lim[0]) & (x <= lim[1])], bins=bins, color="steelblue", alpha=0.8)
    ax_right = ax.inset_axes([1.0, 0, 0.15, 1], sharey=ax)
    ax_right.hist(y[(y >= lim[0]) & (y <= lim[1])], bins=bins, color="steelblue", alpha=0.8,
                  orientation="horizontal")
    for a in (ax_top, ax_right):
        a.axis("off")

    # Summary statistics (computed on all valid objects, not only those inside the plot limits)
    r = np.corrcoef(x, y)[0, 1]
    diff = y - x
    bias = np.median(diff)
    nmad = 1.4826 * np.median(np.abs(diff - bias))
    return dict(n=len(x), pearson_r=r, median_bias=bias, nmad=nmad)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
summary = {}
for ax, p in zip(axes.ravel(), PANELS):
    x, y = valid(p["col"])
    summary[p["col"]] = scatter_panel(ax, x, y, p["label"], p["limits"])
fig.subplots_adjust(wspace=0.45, hspace=0.4)
plt.show()

print(f"{'':6s} {'N':>7s} {'r':>7s} {'med. bias':>11s} {'NMAD':>8s}")
for col, s in summary.items():
    print(f"{col:6s} {s['n']:7d} {s['pearson_r']:7.3f} {s['median_bias']:+11.4f} {s['nmad']:8.4f}")

In [ ]:
# fig.savefig("real_vs_reconstruction_morphology.pdf", bbox_inches="tight")